# 📈 Project 3 — Agentic Sales Agent

An autonomous **B2B sales research and lead-qualification assistant**.

Runs 100% in Google Colab on the **Gemini free tier**. No OpenAI, no paid APIs,
no LangChain/LangGraph/CrewAI/AutoGen, no Docker, no local server.

This is a **sales research assistant**, not an autonomous salesperson: it never
sends outreach, never collects private personal information, and every lead
score is backed by public evidence (or explicitly marked `unknown`). Human
review is required before any sales outreach.

**Workflow:**

```
ICP → ICP Analyzer → Planner → Market Search → Company Discovery
    → Company Research → Qualification → Lead Scoring
    → Prioritization → Final Report
```

Run the cells top to bottom. You'll need a free Gemini API key stored as a
Colab Secret named `GEMINI_API_KEY` (Tools → Secrets).

In [ ]:
# Cell 2 — Install dependencies (Gemini SDK, free web search, lightweight scraping)
!pip install -q google-genai ddgs requests beautifulsoup4

In [ ]:
# Cell 3 — Imports
import os
import json
import zipfile
from pathlib import Path

print("Imports ready.")

In [ ]:
# Cell 4 — Load the Gemini API key from Google Colab Secrets
# Tools -> Secrets -> add a secret named GEMINI_API_KEY, grant this notebook access.
# The key is never hard-coded and never printed.

GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if GEMINI_API_KEY:
    print("✓ Gemini API key loaded (length:", len(GEMINI_API_KEY), "chars, not shown)")
else:
    print("⚠ No GEMINI_API_KEY found yet. Add it under Tools > Secrets before running the agent.")

In [ ]:
# Cell 5 — Generate config.py
# All tunable limits live here so the agent stays inside the Gemini free tier
# and is polite to public websites, and paced/budgeted to avoid free-tier rate limits.

Path("config.py").write_text('"""\nconfig.py — Central configuration for the Agentic Sales Agent.\n\nAll limits here exist to keep the agent inside the Gemini free tier\nand to avoid hammering public websites. Change them with care.\n"""\n\nimport os\n\n# ---------------------------------------------------------------------------\n# Gemini model\n# ---------------------------------------------------------------------------\n# Configurable via environment variable so the same code works if the user\'s\n# account has access to a different Gemini model name. If you see a\n# "model not found" error, change this value (or set GEMINI_MODEL in your\n# environment / Colab Secrets) to a model available to your API key.\nGEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.0-flash")\n\n# ---------------------------------------------------------------------------\n# Free-tier / politeness limits\n# ---------------------------------------------------------------------------\n# Kept deliberately small so a full run (ICP analysis + company discovery +\n# qualification) uses only a handful of Gemini calls, and so the notebook\n# can run several demos back to back without tripping a free-tier API key\'s\n# request-per-minute, request-per-day, or token-per-minute caps.\nMAX_ITERATIONS = 2           # bounded agent loop — never allow infinite execution\nMAX_SEARCH_QUERIES = 3        # number of distinct web searches per run\nMAX_SEARCH_RESULTS = 5        # results kept per search query\nMAX_COMPANIES = 6             # candidate companies carried into research\nMAX_COMPANY_PAGES = 2         # public pages fetched per company\nMAX_PAGE_CHARS = 6_000        # characters kept per fetched page (truncated)\nMAX_COMPANY_EXCERPT_CHARS = 2_500  # per-company text sent to Gemini for qualification\n\nREQUEST_TIMEOUT_SECONDS = 8\nUSER_AGENT = "Mozilla/5.0 (compatible; SalesResearchAgent/1.0; educational-portfolio-project)"\n\n# ---------------------------------------------------------------------------\n# Gemini free-tier request budget & pacing\n# ---------------------------------------------------------------------------\n# Free-tier Gemini API keys are commonly capped at a low number of requests\n# per minute AND a low number of requests per day (some experimental/limited\n# models allow as few as ~20 requests/day). These constants exist so the\n# *whole notebook* — including multiple demo runs — always finishes inside a\n# conservative shared budget, and so calls are paced far enough apart to\n# avoid a per-minute rate limit even when the daily cap isn\'t the issue.\n#\n# Once the session budget is reached, the agent automatically switches to\n# fast, deterministic fallback logic (see sales_agent.py) instead of calling\n# Gemini again, so a run always finishes with a real report instead of\n# crashing or hanging on a 429.\nMAX_GEMINI_CALLS_PER_SESSION = 18   # hard budget; leaves headroom under a 20-request free-tier cap\nGEMINI_MIN_SECONDS_BETWEEN_CALLS = 6.5  # paces calls to stay comfortably under ~9 requests/minute\nGEMINI_MAX_RETRIES = 3\nGEMINI_BACKOFF_SECONDS = 15  # base wait on a rate-limit error; grows with each retry\n\n# ---------------------------------------------------------------------------\n# Lead scoring\n# ---------------------------------------------------------------------------\nSCORING_CRITERIA = [\n    "industry_fit",\n    "geography_fit",\n    "company_size_fit",\n    "problem_fit",\n    "technology_fit",\n    "evidence_quality",\n]\nMAX_CRITERION_SCORE = 3\nMAX_LEAD_SCORE = MAX_CRITERION_SCORE * len(SCORING_CRITERIA)  # 18\n\nPRIORITY_THRESHOLDS = {\n    "HIGH PRIORITY": 15,\n    "MEDIUM PRIORITY": 11,\n    "LOW PRIORITY": 7,\n    # anything below LOW PRIORITY\'s threshold -> INSUFFICIENT DATA\n}\n\n# ---------------------------------------------------------------------------\n# Output paths\n# ---------------------------------------------------------------------------\nOUTPUT_DIR = "outputs"\nLEADS_JSON_PATH = os.path.join(OUTPUT_DIR, "leads.json")\nREPORT_MD_PATH = os.path.join(OUTPUT_DIR, "sales_report.md")\n')
print("✓ config.py written (now with call pacing + a session request budget)")


In [ ]:
# Cell 6 — Generate llm.py (Gemini wrapper)
# Paces calls, enforces a hard session request budget, and backs off hard on
# rate-limit errors so a free-tier key never gets stuck or blows its quota.

Path("llm.py").write_text('"""\nllm.py — Thin wrapper around the Gemini API.\n\nKeeps all Gemini-specific code in one place, tracks call counts (so the\nagent can report how much of the free tier it used), and does its best to\nextract clean JSON from model responses without ever throwing away a run\nbecause of a formatting hiccup.\n\nFree-tier friendliness:\n- Every call is paced at least GEMINI_MIN_SECONDS_BETWEEN_CALLS apart, so\n  the agent doesn\'t burst past a per-minute rate limit.\n- A hard MAX_GEMINI_CALLS_PER_SESSION budget is enforced client-side. Once\n  reached, calls fail fast (no network round trip, no wasted quota) with a\n  GeminiError that callers already know how to catch and fall back from.\n- Rate-limit-shaped errors (429 / quota / resource exhausted) get a much\n  longer backoff than ordinary transient errors, since free-tier limits\n  reset on the order of a minute, not a couple of seconds.\n"""\n\nimport json\nimport re\nimport time\n\nfrom config import (\n    GEMINI_BACKOFF_SECONDS,\n    GEMINI_MAX_RETRIES,\n    GEMINI_MIN_SECONDS_BETWEEN_CALLS,\n    GEMINI_MODEL,\n    MAX_GEMINI_CALLS_PER_SESSION,\n)\n\n\nclass GeminiError(RuntimeError):\n    """Raised when the Gemini API cannot be reached, errors, or the\n    client-side session budget has been used up."""\n\n\n_RATE_LIMIT_MARKERS = (\n    "429",\n    "resource_exhausted",\n    "resource exhausted",\n    "rate limit",\n    "quota",\n)\n\n\nclass GeminiLLM:\n    """\n    Minimal wrapper around google-genai\'s client.\n\n    Usage:\n        llm = GeminiLLM(api_key)\n        text = llm.generate("Write a haiku about SaaS")\n        data = llm.generate_json("Return JSON describing ...")\n    """\n\n    def __init__(self, api_key: str, model: str = GEMINI_MODEL):\n        if not api_key:\n            raise GeminiError(\n                "No Gemini API key was provided. In Colab, add a secret named "\n                "GEMINI_API_KEY (Tools > Secrets) and grant this notebook access."\n            )\n        try:\n            from google import genai\n        except ImportError as exc:\n            raise GeminiError(\n                "google-genai is not installed. Run: pip install -q google-genai"\n            ) from exc\n\n        self._client = genai.Client(api_key=api_key)\n        self.model = model\n        self.call_count = 0\n        self.total_chars_sent = 0\n        self.total_chars_received = 0\n        self._last_call_time = 0.0\n\n    def _wait_for_pacing(self):\n        """Sleep just long enough to keep calls spaced out and avoid RPM limits."""\n        elapsed = time.time() - self._last_call_time\n        remaining = GEMINI_MIN_SECONDS_BETWEEN_CALLS - elapsed\n        if remaining > 0:\n            time.sleep(remaining)\n\n    def generate(self, prompt: str, retries: int = GEMINI_MAX_RETRIES, backoff_seconds: float = GEMINI_BACKOFF_SECONDS) -> str:\n        """Send a single prompt to Gemini and return the text response."""\n        if self.call_count >= MAX_GEMINI_CALLS_PER_SESSION:\n            raise GeminiError(\n                f"Session Gemini call budget reached ({MAX_GEMINI_CALLS_PER_SESSION} calls). "\n                "Skipping this call and using deterministic fallback logic instead, so the "\n                "free-tier key never gets rate-limited or over-quota."\n            )\n\n        last_error = None\n        for attempt in range(retries + 1):\n            self._wait_for_pacing()\n            try:\n                self.call_count += 1\n                self.total_chars_sent += len(prompt)\n                self._last_call_time = time.time()\n                response = self._client.models.generate_content(\n                    model=self.model,\n                    contents=prompt,\n                )\n                text = (getattr(response, "text", None) or "").strip()\n                self.total_chars_received += len(text)\n                if not text:\n                    raise GeminiError("Gemini returned an empty response.")\n                return text\n            except Exception as exc:  # noqa: BLE001 - surface any SDK/network error\n                last_error = exc\n                self._last_call_time = time.time()\n                message = str(exc).lower()\n                # Don\'t burn retries on clearly non-transient errors.\n                if "api key" in message or "permission" in message or "not found" in message:\n                    break\n                if self.call_count >= MAX_GEMINI_CALLS_PER_SESSION:\n                    break\n                if attempt < retries:\n                    is_rate_limit = any(marker in message for marker in _RATE_LIMIT_MARKERS)\n                    wait = backoff_seconds * (attempt + 1) if is_rate_limit else 2.0 * (attempt + 1)\n                    if is_rate_limit:\n                        print(f"  ⏳ Gemini rate limit hit; waiting {wait:.0f}s before retrying...")\n                    time.sleep(wait)\n        raise GeminiError(f"Gemini call failed after retries: {last_error}")\n\n    def generate_json(self, prompt: str, retries: int = GEMINI_MAX_RETRIES) -> dict:\n        """\n        Ask Gemini for JSON and parse it defensively. Gemini sometimes wraps\n        JSON in ```json fences or adds a short preamble — this strips both.\n        """\n        json_prompt = (\n            f"{prompt}\\n\\n"\n            "Respond with ONLY valid JSON. No markdown fences, no preamble, "\n            "no trailing commentary — just the JSON object."\n        )\n        raw = self.generate(json_prompt, retries=retries)\n        return self._extract_json(raw)\n\n    @staticmethod\n    def _extract_json(raw: str) -> dict:\n        cleaned = raw.strip()\n        cleaned = re.sub(r"^```(json)?", "", cleaned.strip(), flags=re.IGNORECASE).strip()\n        cleaned = re.sub(r"```$", "", cleaned.strip()).strip()\n\n        try:\n            return json.loads(cleaned)\n        except json.JSONDecodeError:\n            pass\n\n        # Fall back to grabbing the first {...} block in the text.\n        match = re.search(r"\\{.*\\}", cleaned, flags=re.DOTALL)\n        if match:\n            try:\n                return json.loads(match.group(0))\n            except json.JSONDecodeError:\n                pass\n\n        raise GeminiError(f"Could not parse JSON from Gemini response: {raw[:300]}")\n\n    def stats(self) -> dict:\n        return {\n            "gemini_calls": self.call_count,\n            "gemini_call_budget": MAX_GEMINI_CALLS_PER_SESSION,\n            "gemini_calls_remaining": max(0, MAX_GEMINI_CALLS_PER_SESSION - self.call_count),\n            "chars_sent": self.total_chars_sent,\n            "chars_received": self.total_chars_received,\n        }\n')
print("✓ llm.py written")


In [ ]:
# Cell 7 — Generate tools.py (free web search via ddgs, webpage extraction via
# requests + BeautifulSoup, and normalization/dedupe helpers). These are the
# LOCAL tools the agent uses that do NOT consume Gemini calls.
Path("tools.py").write_text('"""\ntools.py — Local tools the agent uses that do NOT require Gemini:\nfree web search, lightweight webpage fetching/extraction, and small\nnormalization helpers (URL/domain dedup).\n\nKeeping these separate from llm.py is what makes the agent "agentic"\nrather than a single prompt-to-Gemini call: search and extraction are\ndeterministic, inspectable, and free.\n"""\n\nimport re\nimport time\nfrom urllib.parse import urlparse\n\nimport requests\n\nfrom config import (\n    MAX_PAGE_CHARS,\n    MAX_SEARCH_RESULTS,\n    REQUEST_TIMEOUT_SECONDS,\n    USER_AGENT,\n)\n\n\nclass SearchError(RuntimeError):\n    """Raised when the search backend fails entirely (not just zero results)."""\n\n\ndef web_search(query: str, max_results: int = MAX_SEARCH_RESULTS) -> list:\n    """\n    Free web search using ddgs (DuckDuckGo Search).\n    Returns a list of {"title", "url", "snippet"} dicts. Never raises for\n    "no results" — only for a hard backend failure, and even then it\n    returns an empty list so the agent can continue.\n    """\n    try:\n        from ddgs import DDGS\n    except ImportError:\n        try:\n            from duckduckgo_search import DDGS  # older package name, fallback\n        except ImportError as exc:\n            raise SearchError(\n                "No search backend installed. Run: pip install -q ddgs"\n            ) from exc\n\n    results = []\n    try:\n        with DDGS() as ddgs:\n            for item in ddgs.text(query, max_results=max_results):\n                results.append(\n                    {\n                        "title": (item.get("title") or "").strip(),\n                        "url": (item.get("href") or item.get("url") or "").strip(),\n                        "snippet": (item.get("body") or item.get("snippet") or "").strip(),\n                    }\n                )\n    except Exception as exc:  # noqa: BLE001\n        print(f"  ⚠ search failed for query \'{query}\': {exc}")\n        return []\n\n    return results[:max_results]\n\n\ndef normalize_domain(url: str) -> str:\n    """Extract a comparable root domain from a URL, e.g. \'www.acme.com\' -> \'acme.com\'."""\n    if not url:\n        return ""\n    try:\n        netloc = urlparse(url if "://" in url else f"https://{url}").netloc.lower()\n        netloc = re.sub(r"^www\\.", "", netloc)\n        return netloc\n    except Exception:  # noqa: BLE001\n        return url.lower().strip()\n\n\ndef normalize_company_name(name: str) -> str:\n    """Lowercase + strip common suffixes so \'Acme Inc.\' and \'Acme\' can be matched."""\n    if not name:\n        return ""\n    cleaned = name.strip().lower()\n    cleaned = re.sub(r"[.,]", "", cleaned)\n    cleaned = re.sub(\n        r"\\b(inc|ltd|llc|pvt|private|limited|corp|corporation|co)\\b", "", cleaned\n    )\n    cleaned = re.sub(r"\\s+", " ", cleaned).strip()\n    return cleaned\n\n\ndef fetch_page_text(url: str, max_chars: int = MAX_PAGE_CHARS) -> str:\n    """\n    Fetch a single public webpage and return cleaned, truncated text.\n    Returns an empty string (never raises) on any failure — timeouts,\n    blocks, non-HTML content, etc. — so the agent can keep going with\n    partial information rather than crashing.\n    """\n    try:\n        from bs4 import BeautifulSoup\n    except ImportError:\n        print("  ⚠ beautifulsoup4 not installed; skipping page extraction")\n        return ""\n\n    headers = {"User-Agent": USER_AGENT}\n    try:\n        resp = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)\n        resp.raise_for_status()\n        content_type = resp.headers.get("Content-Type", "")\n        if "html" not in content_type.lower():\n            return ""\n\n        soup = BeautifulSoup(resp.text, "html.parser")\n        for tag in soup(["script", "style", "noscript", "svg", "footer", "nav"]):\n            tag.decompose()\n\n        text = soup.get_text(separator=" ")\n        text = re.sub(r"\\s+", " ", text).strip()\n        return text[:max_chars]\n    except requests.exceptions.RequestException as exc:\n        print(f"  ⚠ could not fetch {url}: {exc}")\n        return ""\n    except Exception as exc:  # noqa: BLE001\n        print(f"  ⚠ unexpected error fetching {url}: {exc}")\n        return ""\n\n\ndef polite_pause(seconds: float = 0.5):\n    """Small delay between outbound requests to be a good web citizen."""\n    time.sleep(seconds)\n\n\ndef dedupe_companies(companies: list) -> list:\n    """\n    Remove duplicate candidate companies by normalized name OR normalized\n    domain, keeping the first (best-ranked) occurrence and merging any\n    extra source URLs into it.\n    """\n    seen_names = {}\n    seen_domains = {}\n    deduped = []\n\n    for company in companies:\n        name_key = normalize_company_name(company.get("company", ""))\n        domain_key = normalize_domain(company.get("website", ""))\n\n        existing_index = seen_names.get(name_key) if name_key else None\n        if existing_index is None and domain_key:\n            existing_index = seen_domains.get(domain_key)\n\n        if existing_index is not None:\n            existing = deduped[existing_index]\n            for url in company.get("source_urls", []):\n                if url not in existing["source_urls"]:\n                    existing["source_urls"].append(url)\n            continue\n\n        deduped.append(company)\n        index = len(deduped) - 1\n        if name_key:\n            seen_names[name_key] = index\n        if domain_key:\n            seen_domains[domain_key] = index\n\n    return deduped\n')
print("✓ tools.py written")

In [ ]:
# Cell 8 — Generate scoring.py
# Deterministic, unit-testable lead scoring — NOT delegated to the LLM, so
# business scoring stays transparent and reproducible.
Path("scoring.py").write_text('"""\nscoring.py — Deterministic lead scoring.\n\nThis is intentionally plain Python, not an LLM call: business scoring\nshould be transparent, reproducible, and auditable. Gemini is used to\n*propose* per-criterion scores with evidence (see sales_agent.py\'s\nqualification step), but the arithmetic and priority classification\nthat turns those scores into a decision live here, where they can be\nunit tested and never hallucinated.\n"""\n\nfrom config import MAX_CRITERION_SCORE, MAX_LEAD_SCORE, PRIORITY_THRESHOLDS, SCORING_CRITERIA\n\n\nclass InvalidScoreError(ValueError):\n    """Raised when a criterion score is outside the allowed range."""\n\n\ndef validate_criterion_score(value) -> int:\n    """\n    A criterion score must be an int in [0, MAX_CRITERION_SCORE], or the\n    string "unknown" (treated as 0 for scoring, but preserved as evidence\n    that the agent found no information rather than guessing).\n    """\n    if value == "unknown":\n        return 0\n    if not isinstance(value, int) or isinstance(value, bool):\n        raise InvalidScoreError(f"Score must be an int or \'unknown\', got: {value!r}")\n    if not (0 <= value <= MAX_CRITERION_SCORE):\n        raise InvalidScoreError(\n            f"Score {value} out of range 0-{MAX_CRITERION_SCORE}"\n        )\n    return value\n\n\ndef calculate_lead_score(criteria: dict) -> int:\n    """\n    criteria: dict mapping each of SCORING_CRITERIA -> {"score": int|"unknown", "evidence": str}\n    Returns the total lead score (int, 0-MAX_LEAD_SCORE).\n    Missing criteria are treated as score 0 ("unknown").\n    """\n    total = 0\n    for name in SCORING_CRITERIA:\n        entry = criteria.get(name, {})\n        raw_score = entry.get("score", "unknown") if isinstance(entry, dict) else "unknown"\n        total += validate_criterion_score(raw_score)\n    return total\n\n\ndef classify_priority(score: int) -> str:\n    """Map a total score to a priority bucket using PRIORITY_THRESHOLDS."""\n    if not isinstance(score, int) or isinstance(score, bool):\n        raise InvalidScoreError(f"Total score must be an int, got: {score!r}")\n    if not (0 <= score <= MAX_LEAD_SCORE):\n        raise InvalidScoreError(f"Total score {score} out of range 0-{MAX_LEAD_SCORE}")\n\n    if score >= PRIORITY_THRESHOLDS["HIGH PRIORITY"]:\n        return "HIGH PRIORITY"\n    if score >= PRIORITY_THRESHOLDS["MEDIUM PRIORITY"]:\n        return "MEDIUM PRIORITY"\n    if score >= PRIORITY_THRESHOLDS["LOW PRIORITY"]:\n        return "LOW PRIORITY"\n    return "INSUFFICIENT DATA"\n\n\ndef score_lead(company: str, criteria: dict, sources: list = None) -> dict:\n    """\n    Build the full scored-lead record for one company.\n    Raises InvalidScoreError if any criterion score is malformed — the\n    caller should catch this and mark the lead INSUFFICIENT DATA rather\n    than crash the whole run.\n    """\n    total = calculate_lead_score(criteria)\n    return {\n        "company": company,\n        "score": total,\n        "max_score": MAX_LEAD_SCORE,\n        "priority": classify_priority(total),\n        "criteria": criteria,\n        "sources": sources or [],\n    }\n')
print("✓ scoring.py written")

In [ ]:
# Cell 9 — Generate sales_agent.py
# Contains: SalesState (agent state), ICP analyzer, planner, market search,
# company discovery, company research, qualification, the bounded agent
# loop, and the final Markdown report generator.
# Each loop pass now only discovers/researches/qualifies NEW companies —
# already-processed ones are never re-sent to Gemini — and qualification
# falls back to fast deterministic keyword scoring if Gemini errors out or
# the session's free-tier call budget is reached.

Path("sales_agent.py").write_text('"""\nsales_agent.py — The Agentic B2B Sales Research Agent.\n\nThis is a research and lead-qualification assistant, NOT an autonomous\nsalesperson. It never contacts anyone, never collects private personal\ninformation, and never invents evidence. Every score is backed by a\npiece of public text the agent actually retrieved, or explicitly marked\n"unknown". Human review is required before any outreach.\n\nArchitecture:\n\n    ICP -> ICP ANALYZER -> PLANNER -> MARKET SEARCH -> COMPANY DISCOVERY\n        -> COMPANY RESEARCH -> QUALIFICATION -> LEAD SCORING\n        -> PRIORITIZATION -> FINAL REPORT\n\nThe agent runs a bounded loop (MAX_ITERATIONS) that decides, at each\nstep, whether it has enough evidence to finish or needs another round\nof research. Each pass only sends *new* companies to Gemini — companies\nalready discovered/researched/qualified in an earlier pass are never\nre-sent, which keeps the whole run comfortably inside a free-tier\nGemini API key\'s request budget (see config.py).\n"""\n\nimport json\nfrom dataclasses import dataclass, field\nfrom datetime import datetime, timezone\n\nfrom config import (\n    MAX_COMPANIES,\n    MAX_COMPANY_EXCERPT_CHARS,\n    MAX_COMPANY_PAGES,\n    MAX_ITERATIONS,\n    MAX_SEARCH_QUERIES,\n)\nfrom llm import GeminiError, GeminiLLM\nfrom scoring import InvalidScoreError, score_lead\nfrom tools import (\n    dedupe_companies,\n    fetch_page_text,\n    normalize_domain,\n    polite_pause,\n    web_search,\n)\n\nVALID_ACTIONS = [\n    "ANALYZE_ICP",\n    "GENERATE_SEARCH_QUERIES",\n    "SEARCH_MARKET",\n    "RESEARCH_COMPANY",\n    "QUALIFY_LEADS",\n    "IDENTIFY_GAPS",\n    "FINISH",\n]\n\nRESEARCH_PAGE_PATHS = ["", "/about", "/product", "/solutions", "/technology", "/industries", "/blog"]\n\n\n# ---------------------------------------------------------------------------\n# Agent state\n# ---------------------------------------------------------------------------\n@dataclass\nclass SalesState:\n    icp_raw: dict\n    icp_analysis: dict = field(default_factory=dict)\n    plan: list = field(default_factory=list)\n    search_queries: list = field(default_factory=list)\n    search_results: list = field(default_factory=list)\n    candidate_companies: list = field(default_factory=list)\n    researched_companies: list = field(default_factory=list)\n    qualified_leads: list = field(default_factory=list)\n    insufficient_leads: list = field(default_factory=list)\n    gaps: list = field(default_factory=list)\n    iterations: int = 0\n    action_log: list = field(default_factory=list)\n    final_report: str = ""\n    # Internal bookkeeping so each loop pass only processes *new* work\n    # (new search results / new candidates) instead of re-sending\n    # already-processed companies to Gemini on every iteration.\n    discovered_result_count: int = 0\n\n    def log(self, action: str, detail: str = ""):\n        self.action_log.append({"iteration": self.iterations, "action": action, "detail": detail})\n\n\n# ---------------------------------------------------------------------------\n# The agent\n# ---------------------------------------------------------------------------\nclass SalesAgent:\n    def __init__(self, llm: GeminiLLM, verbose: bool = True):\n        self.llm = llm\n        self.verbose = verbose\n        self.search_call_count = 0\n        self.companies_researched_count = 0\n\n    def _say(self, message: str):\n        if self.verbose:\n            print(message)\n\n    # -- ANALYZE_ICP ---------------------------------------------------\n    def analyze_icp(self, state: SalesState) -> SalesState:\n        self._say("\\n🧠 Analyzing ICP...")\n        prompt = f"""\nYou are a B2B sales research strategist. Turn this raw Ideal Customer\nProfile (ICP) into a structured research strategy.\n\nRaw ICP:\n{json.dumps(state.icp_raw, indent=2)}\n\nReturn a JSON object with exactly these keys:\n- "target_industry": string\n- "target_geography": string\n- "company_size": string\n- "ideal_customer_problem": string\n- "relevant_signals": array of short strings (public signals that would\n  suggest a company fits this ICP, e.g. "publicly discusses manual\n  processes", "recent hiring for automation roles")\n- "search_terms": array of 3-5 short search-engine-style query strings\n  (not full sentences) likely to surface real candidate companies\n- "qualification_criteria": array of short strings describing what\n  would count as strong evidence of fit\n- "disqualification_criteria": array of short strings describing what\n  would rule a company out\n"""\n        try:\n            analysis = self.llm.generate_json(prompt)\n        except GeminiError as exc:\n            self._say(f"  ⚠ ICP analysis failed ({exc}); falling back to the raw ICP.")\n            analysis = self._fallback_icp_analysis(state.icp_raw)\n\n        state.icp_analysis = analysis\n        state.plan = [\n            "Generate search queries from ICP",\n            "Search public web for candidate companies",\n            "Research top candidates from public pages",\n            "Qualify and score leads against ICP",\n            "Identify information gaps",\n            "Produce final research report",\n        ]\n        state.log("ANALYZE_ICP", "ICP structured into research strategy")\n        self._say("✓ ICP analyzed")\n        return state\n\n    @staticmethod\n    def _fallback_icp_analysis(icp_raw: dict) -> dict:\n        """Deterministic fallback if Gemini is unavailable, so the agent can still run."""\n        industry = icp_raw.get("industry", "")\n        geography = icp_raw.get("geography", "")\n        size = icp_raw.get("company_size", "")\n        return {\n            "target_industry": industry,\n            "target_geography": geography,\n            "company_size": size,\n            "ideal_customer_problem": icp_raw.get("customer_problem", ""),\n            "relevant_signals": ["publicly discusses relevant operational challenges"],\n            "search_terms": [\n                f"{industry} companies {geography}",\n                f"{industry} startups {geography} {size} employees",\n            ],\n            "qualification_criteria": ["industry match", "geography match", "size match"],\n            "disqualification_criteria": ["unrelated industry", "no public presence"],\n        }\n\n    # -- GENERATE_SEARCH_QUERIES ----------------------------------------\n    def generate_search_queries(self, state: SalesState) -> SalesState:\n        self._say("\\n📋 Creating search strategy...")\n        terms = state.icp_analysis.get("search_terms") or []\n        queries = [t for t in terms if isinstance(t, str) and t.strip()][:MAX_SEARCH_QUERIES]\n        if not queries:\n            queries = [\n                f"{state.icp_analysis.get(\'target_industry\', \'\')} companies "\n                f"{state.icp_analysis.get(\'target_geography\', \'\')}"\n            ]\n        state.search_queries = queries\n        state.log("GENERATE_SEARCH_QUERIES", f"{len(queries)} queries created")\n        self._say(f"✓ {len(queries)} search queries created")\n        return state\n\n    # -- SEARCH_MARKET ---------------------------------------------------\n    def search_market(self, state: SalesState) -> SalesState:\n        for i, query in enumerate(state.search_queries, start=1):\n            self._say(f"\\n🔎 Market Search {i}/{len(state.search_queries)}: \\"{query}\\"")\n            results = web_search(query)\n            self.search_call_count += 1\n            state.search_results.extend(results)\n            self._say(f"✓ {len(results)} results found")\n            polite_pause()\n        state.log("SEARCH_MARKET", f"{len(state.search_results)} total results collected")\n        return state\n\n    # -- COMPANY DISCOVERY ------------------------------------------------\n    def discover_companies(self, state: SalesState) -> SalesState:\n        """Extract candidate companies from any search results not yet\n        processed. Safe to call every loop pass: it\'s a no-op (and makes\n        no Gemini call) once there\'s nothing new to look at."""\n        new_results = state.search_results[state.discovered_result_count:]\n        if not new_results:\n            state.log("RESEARCH_COMPANY", "no new search results to extract companies from")\n            return state\n\n        remaining_slots = MAX_COMPANIES - len(state.candidate_companies)\n        if remaining_slots <= 0:\n            state.discovered_result_count = len(state.search_results)\n            state.log("RESEARCH_COMPANY", "candidate list already full; skipping discovery call")\n            return state\n\n        self._say("\\n🏢 Extracting candidate companies from search results...")\n\n        # Batch all new search results into a single Gemini call to extract\n        # structured company candidates (keeps LLM usage low).\n        results_text = "\\n".join(\n            f"- title: {r[\'title\']} | url: {r[\'url\']} | snippet: {r[\'snippet\']}"\n            for r in new_results\n            if r.get("url")\n        )\n        prompt = f"""\nFrom these web search results, extract distinct real companies that\ncould plausibly be evaluated as B2B sales leads. Ignore directories,\nnews aggregators, "top 10 lists" articles (unless a specific company\nis clearly named), and non-company pages.\n\nSearch results:\n{results_text}\n\nReturn a JSON object with one key "companies", an array of up to\n{remaining_slots} objects, each with:\n- "company": company name\n- "website": best-guess homepage URL (from the results, or your best\n  inference from the domain in the result URL)\n- "source_urls": array of the result URLs that mention this company\n- "discovery_reason": one short sentence on why this looked like a\n  plausible candidate given the search context\n"""\n        try:\n            data = self.llm.generate_json(prompt)\n            new_companies = data.get("companies", [])\n        except GeminiError as exc:\n            self._say(f"  ⚠ company extraction failed ({exc}); using raw search results as candidates.")\n            new_companies = self._fallback_discovery(new_results)\n\n        combined = dedupe_companies(state.candidate_companies + new_companies)[:MAX_COMPANIES]\n        state.candidate_companies = combined\n        state.discovered_result_count = len(state.search_results)\n        state.log("RESEARCH_COMPANY", f"{len(combined)} candidate companies discovered so far")\n        self._say(f"✓ Candidate companies discovered: {len(combined)}")\n        return state\n\n    @staticmethod\n    def _fallback_discovery(search_results: list) -> list:\n        """Deterministic fallback: treat each unique domain as a candidate."""\n        seen = set()\n        companies = []\n        for r in search_results:\n            domain = normalize_domain(r.get("url", ""))\n            if not domain or domain in seen:\n                continue\n            seen.add(domain)\n            companies.append(\n                {\n                    "company": r.get("title", domain).split("|")[0].strip(),\n                    "website": r.get("url", ""),\n                    "source_urls": [r.get("url", "")],\n                    "discovery_reason": "Appeared in market search results (automated fallback, no Gemini call used).",\n                }\n            )\n        return companies[:MAX_COMPANIES]\n\n    # -- COMPANY RESEARCH ---------------------------------------------------\n    def research_companies(self, state: SalesState) -> SalesState:\n        """Fetch public pages for any candidate companies not yet researched.\n        Uses only local, free HTTP requests — no Gemini calls."""\n        already = {c.get("company") for c in state.researched_companies}\n        pending = [c for c in state.candidate_companies if c.get("company") not in already]\n        if not pending:\n            state.log("RESEARCH_COMPANY", "no new candidate companies to research")\n            return state\n\n        for company in pending:\n            name = company.get("company", "unknown company")\n            website = company.get("website", "")\n            self._say(f"\\n🔍 Researching: {name}")\n\n            pages_text = []\n            if website:\n                base = website.rstrip("/")\n                for path in RESEARCH_PAGE_PATHS[:MAX_COMPANY_PAGES]:\n                    url = base + path\n                    text = fetch_page_text(url)\n                    if text:\n                        pages_text.append({"url": url, "text": text})\n                    polite_pause()\n\n            self.companies_researched_count += 1\n            researched = {\n                **company,\n                "public_pages": pages_text,\n                "has_public_data": bool(pages_text),\n            }\n            state.researched_companies.append(researched)\n\n            if pages_text:\n                self._say("✓ Public company information collected")\n            else:\n                self._say("  ⚠ no public page content retrieved (site unreachable or blocked)")\n\n        state.log("RESEARCH_COMPANY", f"{len(state.researched_companies)} companies researched so far")\n        return state\n\n    # -- QUALIFY_LEADS ---------------------------------------------------\n    def qualify_leads(self, state: SalesState) -> SalesState:\n        """Score any researched companies not yet qualified. Skips the\n        Gemini call entirely (and logs why) when there\'s nothing new,\n        which is what prevents the agentic loop from re-scoring the same\n        companies — and burning the same request budget — on every pass."""\n        already_seen = {l["company"] for l in state.qualified_leads} | {\n            i["company"] for i in state.insufficient_leads\n        }\n        pending = [c for c in state.researched_companies if c.get("company") not in already_seen]\n\n        if not pending:\n            if not state.researched_companies:\n                state.log("QUALIFY_LEADS", "no researched companies to qualify")\n            else:\n                state.log("QUALIFY_LEADS", "no new companies since last pass; skipping duplicate Gemini call")\n            return state\n\n        self._say("\\n📊 Qualifying leads...")\n\n        # One batched Gemini call scores all newly researched companies together.\n        companies_payload = []\n        for c in pending:\n            combined_text = " ".join(p["text"] for p in c.get("public_pages", []))[:MAX_COMPANY_EXCERPT_CHARS]\n            companies_payload.append(\n                {\n                    "company": c.get("company"),\n                    "website": c.get("website"),\n                    "public_text_excerpt": combined_text or "(no public page text retrieved)",\n                    "source_urls": c.get("source_urls", []),\n                }\n            )\n\n        prompt = f"""\nYou are qualifying B2B sales leads against this ICP:\n{json.dumps(state.icp_analysis, indent=2)}\n\nFor each company below, score these six criteria on a 0-3 scale:\n0 = no evidence, 1 = weak, 2 = moderate, 3 = strong.\nCriteria: industry_fit, geography_fit, company_size_fit, problem_fit,\ntechnology_fit, evidence_quality.\n\nCRITICAL RULES:\n- Base every score ONLY on the provided public_text_excerpt or\n  source_urls. Do not invent facts about the company.\n- If the excerpt does not support a criterion, set that criterion\'s\n  "score" to the string "unknown" rather than guessing a number.\n- Every criterion needs a one-sentence "evidence" string quoting or\n  paraphrasing what in the text supports the score, or "no public\n  evidence found" if unknown.\n- Suggest a cautious "sales_angle" (one sentence, phrased as a\n  possibility, e.g. "Explore whether X could help with Y") ONLY if\n  there is supporting evidence; otherwise use "insufficient evidence\n  for a sales angle".\n\nCompanies:\n{json.dumps(companies_payload, indent=2)}\n\nReturn a JSON object with key "qualifications": an array, one entry\nper company, each with:\n{{\n  "company": "...",\n  "criteria": {{\n     "industry_fit": {{"score": 0-3 or "unknown", "evidence": "..."}},\n     "geography_fit": {{...}},\n     "company_size_fit": {{...}},\n     "problem_fit": {{...}},\n     "technology_fit": {{...}},\n     "evidence_quality": {{...}}\n  }},\n  "sales_angle": "...",\n  "missing_information": ["..."]\n}}\n"""\n        try:\n            data = self.llm.generate_json(prompt)\n            qualifications = data.get("qualifications", [])\n        except GeminiError as exc:\n            self._say(f"  ⚠ qualification call failed ({exc}); using automated keyword-fallback scoring.")\n            qualifications = self._fallback_qualify(companies_payload, state.icp_analysis)\n\n        qual_by_company = {q.get("company"): q for q in qualifications}\n\n        for c in pending:\n            name = c.get("company")\n            qual = qual_by_company.get(name)\n            sources = c.get("source_urls", [])\n            if not qual:\n                state.insufficient_leads.append(\n                    {\n                        "company": name,\n                        "website": c.get("website", ""),\n                        "reason": "No qualification data returned for this company.",\n                        "sources": sources,\n                    }\n                )\n                continue\n            try:\n                lead = score_lead(name, qual.get("criteria", {}), sources)\n                lead["website"] = c.get("website", "")\n                lead["sales_angle"] = qual.get("sales_angle", "insufficient evidence for a sales angle")\n                lead["missing_information"] = qual.get("missing_information", [])\n                lead["discovery_reason"] = c.get("discovery_reason", "")\n                state.qualified_leads.append(lead)\n            except InvalidScoreError as exc:\n                state.insufficient_leads.append(\n                    {\n                        "company": name,\n                        "website": c.get("website", ""),\n                        "reason": f"Malformed scoring data: {exc}",\n                        "sources": sources,\n                    }\n                )\n\n        state.qualified_leads.sort(key=lambda lead: lead["score"], reverse=True)\n        state.log("QUALIFY_LEADS", f"{len(pending)} new companies scored this pass ({len(state.qualified_leads)} total leads)")\n        self._say("✓ Lead score generated" if state.qualified_leads else "⚠ no leads could be scored")\n        return state\n\n    @staticmethod\n    def _fallback_qualify(companies_payload: list, icp_analysis: dict) -> list:\n        """Deterministic keyword-matching fallback used when Gemini is\n        unavailable (API error, or the session\'s free-tier call budget has\n        been used up). It never invents facts: criteria are only ever\n        "unknown", 1 (weak keyword match), or 2 (direct keyword match) —\n        never a confident 3 — and every evidence string says plainly that\n        this is automated fallback scoring, not a Gemini judgement. This is\n        what lets the notebook always finish with a real, populated report\n        even if the free-tier key runs out of quota mid-run."""\n        industry = str(icp_analysis.get("target_industry", "")).lower().strip()\n        geography = str(icp_analysis.get("target_geography", "")).lower().strip()\n        problem_words = [\n            w for w in str(icp_analysis.get("ideal_customer_problem", "")).lower().split()\n            if len(w) > 4\n        ]\n        signal_words = [str(s).lower() for s in icp_analysis.get("relevant_signals", []) if s]\n\n        def crit(has_text: bool, hit: bool) -> dict:\n            if not has_text:\n                return {"score": "unknown", "evidence": "no public evidence found"}\n            if hit:\n                return {"score": 2, "evidence": "keyword match found in public page text (automated fallback scoring, no Gemini call used)"}\n            return {"score": "unknown", "evidence": "public text present but no direct keyword match (automated fallback scoring)"}\n\n        qualifications = []\n        for c in companies_payload:\n            text = str(c.get("public_text_excerpt", "")).lower()\n            has_text = bool(text) and text != "(no public page text retrieved)"\n            qualifications.append(\n                {\n                    "company": c.get("company"),\n                    "criteria": {\n                        "industry_fit": crit(has_text, bool(industry) and industry in text),\n                        "geography_fit": crit(has_text, bool(geography) and geography in text),\n                        "company_size_fit": {"score": "unknown", "evidence": "company size is not reliably inferable without a Gemini analysis pass"},\n                        "problem_fit": crit(has_text, any(w in text for w in problem_words)),\n                        "technology_fit": crit(has_text, any(w in text for w in signal_words)),\n                        "evidence_quality": crit(has_text, has_text),\n                    },\n                    "sales_angle": "insufficient evidence for a sales angle",\n                    "missing_information": [\n                        "Full qualification unavailable this run — Gemini free-tier budget was reached, "\n                        "so this company was scored with automated keyword matching instead."\n                    ],\n                }\n            )\n        return qualifications\n\n    # -- IDENTIFY_GAPS ---------------------------------------------------\n    def identify_gaps(self, state: SalesState) -> bool:\n        """Returns True if evidence is sufficient to finish, False if another loop is warranted."""\n        self._say("\\n🔍 Checking information gaps...")\n        gaps = []\n        if not state.qualified_leads:\n            gaps.append("No leads were successfully scored.")\n        strong_leads = [l for l in state.qualified_leads if l["priority"] in ("HIGH PRIORITY", "MEDIUM PRIORITY")]\n        if not strong_leads and state.iterations < MAX_ITERATIONS - 1:\n            gaps.append("No HIGH/MEDIUM priority leads yet found.")\n        state.gaps = gaps\n        state.log("IDENTIFY_GAPS", "; ".join(gaps) if gaps else "sufficient evidence")\n\n        if not gaps:\n            self._say("✓ Sufficient evidence collected")\n            return True\n\n        if state.iterations >= MAX_ITERATIONS - 1:\n            self._say(f"✓ Reached max iterations ({MAX_ITERATIONS}); finishing with available evidence")\n            return True\n\n        self._say(f"⚠ Gaps found: {gaps}. Will attempt additional research.")\n        return False\n\n    # -- FINAL REPORT ---------------------------------------------------\n    def generate_report(self, state: SalesState) -> SalesState:\n        self._say("\\n🧠 Generating final sales research report...")\n        stats = self.llm.stats()\n        stats["search_calls"] = self.search_call_count\n        stats["companies_discovered"] = len(state.candidate_companies)\n        stats["companies_researched"] = self.companies_researched_count\n        stats["agent_iterations"] = state.iterations\n\n        report = build_markdown_report(state, stats)\n        state.final_report = report\n        state.log("FINISH", "final report generated")\n        self._say("✓ Report generated")\n        return state\n\n    # -- AGENT LOOP ---------------------------------------------------\n    def run(self, icp_raw: dict) -> SalesState:\n        state = SalesState(icp_raw=icp_raw)\n\n        self._say("=" * 56)\n        self._say("🤖 AGENTIC SALES AGENT")\n        self._say("=" * 56)\n        self._say(f"\\n🎯 Ideal Customer Profile\\n")\n        for key, value in icp_raw.items():\n            self._say(f"{key.replace(\'_\', \' \').title()}: {value}")\n\n        state = self.analyze_icp(state)\n        state = self.generate_search_queries(state)\n\n        done = False\n        while state.iterations < MAX_ITERATIONS and not done:\n            state.iterations += 1\n            self._say(f"\\n--- Agent iteration {state.iterations}/{MAX_ITERATIONS} ---")\n\n            if not state.search_results:\n                state = self.search_market(state)\n            state = self.discover_companies(state)\n            state = self.research_companies(state)\n            state = self.qualify_leads(state)\n            done = self.identify_gaps(state)\n\n            if not done:\n                # Extra loop: broaden with the next unused search term, if\n                # any, so the *next* pass has genuinely new search results\n                # to discover/research/qualify from — otherwise stop to\n                # respect MAX_ITERATIONS.\n                extra_terms = state.icp_analysis.get("search_terms", [])\n                used = set(state.search_queries)\n                remaining = [t for t in extra_terms if t not in used]\n                if remaining:\n                    query = remaining[0]\n                    state.search_queries.append(query)\n                    self._say(f"\\n🔎 Broadening search: \\"{query}\\"")\n                    extra_results = web_search(query)\n                    self.search_call_count += 1\n                    state.search_results.extend(extra_results)\n                    polite_pause()\n                else:\n                    done = True\n\n        state = self.generate_report(state)\n\n        self._say("\\n" + "=" * 56)\n        self._say("📈 Run complete. See state.final_report / outputs/ for details.")\n        self._say("=" * 56)\n        return state\n\n\n# ---------------------------------------------------------------------------\n# Report generation (Markdown)\n# ---------------------------------------------------------------------------\ndef build_markdown_report(state: SalesState, stats: dict) -> str:\n    icp = state.icp_analysis or state.icp_raw\n    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")\n\n    lines = []\n    lines.append("# 📈 Sales Research Report")\n    lines.append(f"_Generated {now}_\\n")\n\n    lines.append("## Executive Summary")\n    high = [l for l in state.qualified_leads if l["priority"] == "HIGH PRIORITY"]\n    medium = [l for l in state.qualified_leads if l["priority"] == "MEDIUM PRIORITY"]\n    lines.append(\n        f"This report evaluates {len(state.researched_companies)} publicly "\n        f"discoverable companies against the target ICP. "\n        f"{len(high)} were classified HIGH PRIORITY and {len(medium)} MEDIUM "\n        f"PRIORITY. All scores are derived from public web content only and "\n        f"require human review before any outreach.\\n"\n    )\n\n    lines.append("## Ideal Customer Profile")\n    for key, value in icp.items():\n        if isinstance(value, list):\n            value = ", ".join(str(v) for v in value)\n        lines.append(f"- **{key.replace(\'_\', \' \').title()}**: {value}")\n    lines.append("")\n\n    lines.append("## Research Methodology")\n    lines.append(\n        "1. The raw ICP was analyzed by Gemini into a structured research strategy.\\n"\n        "2. Multiple targeted web searches (DuckDuckGo, free tier) were run to surface candidate companies.\\n"\n        "3. Public company pages (homepage, about, product, etc.) were fetched for top candidates.\\n"\n        "4. Each company was scored against six criteria using only retrieved public text, with \'unknown\' used instead of guessing where evidence was absent.\\n"\n        "5. Deterministic Python scoring (see `scoring.py`) converted per-criterion scores into a total and a priority bucket.\\n"\n    )\n\n    lines.append("## Top Qualified Leads")\n    if not state.qualified_leads:\n        lines.append("_No leads met the scoring bar this run. See Limitations below._\\n")\n    else:\n        for lead in state.qualified_leads[:5]:\n            lines.append(f"- **{lead[\'company\']}** — {lead[\'score\']}/{lead[\'max_score\']} ({lead[\'priority\']})")\n        lines.append("")\n\n    lines.append("## Lead Scoring")\n    lines.append(\n        "Score = industry_fit + geography_fit + company_size_fit + problem_fit + "\n        "technology_fit + evidence_quality (each 0-3, max 18).\\n"\n        "- 15-18 → HIGH PRIORITY\\n"\n        "- 11-14 → MEDIUM PRIORITY\\n"\n        "- 7-10 → LOW PRIORITY\\n"\n        "- below 7 → INSUFFICIENT DATA\\n"\n    )\n\n    lines.append("## Company-by-Company Analysis")\n    if not state.qualified_leads:\n        lines.append("_No scored leads to display._\\n")\n    for lead in state.qualified_leads:\n        lines.append(f"### {lead[\'company\']}")\n        lines.append(f"- Website: {lead.get(\'website\', \'unknown\')}")\n        lines.append(f"- Score: {lead[\'score\']}/{lead[\'max_score\']} — **{lead[\'priority\']}**")\n        lines.append(f"- Why relevant: {lead.get(\'discovery_reason\', \'n/a\')}")\n        lines.append("- Evidence by criterion:")\n        for crit, entry in lead.get("criteria", {}).items():\n            score = entry.get("score", "unknown")\n            evidence = entry.get("evidence", "unknown")\n            lines.append(f"  - {crit.replace(\'_\', \' \').title()}: {score} — {evidence}")\n        lines.append(f"- Recommended sales angle: {lead.get(\'sales_angle\', \'insufficient evidence for a sales angle\')}")\n        if lead.get("missing_information"):\n            lines.append(f"- Missing information: {\', \'.join(lead[\'missing_information\'])}")\n        if lead.get("sources"):\n            lines.append(f"- Sources: {\', \'.join(lead[\'sources\'])}")\n        lines.append("")\n\n    if state.insufficient_leads:\n        lines.append("## Insufficient Data")\n        for item in state.insufficient_leads:\n            lines.append(f"- **{item[\'company\']}** — {item[\'reason\']}")\n        lines.append("")\n\n    lines.append("## Missing Information")\n    if state.gaps:\n        for gap in state.gaps:\n            lines.append(f"- {gap}")\n    else:\n        lines.append("- None identified for this run.")\n    lines.append("")\n\n    lines.append("## Sources")\n    all_sources = sorted({s for lead in state.qualified_leads for s in lead.get("sources", [])})\n    if all_sources:\n        for s in all_sources:\n            lines.append(f"- {s}")\n    else:\n        lines.append("- No sources recorded.")\n    lines.append("")\n\n    lines.append("## Limitations")\n    lines.append(\n        "- This report reflects only what public web pages and search "\n        "results made available at run time; it is not exhaustive.\\n"\n        "- Company size, revenue, and technology stack are frequently not "\n        "publicly disclosed and are marked \'unknown\' rather than guessed.\\n"\n        "- Scores reflect *apparent* fit based on public marketing "\n        "content, not verified qualification — a human must confirm "\n        "before any outreach.\\n"\n        "- No company listed here has been contacted, and none should be "\n        "treated as having opted in to any communication.\\n"\n        "- If the Gemini free-tier request budget was reached mid-run, "\n        "remaining companies were scored with an automated keyword "\n        "fallback (clearly labeled in \'Missing information\' above) "\n        "instead of a Gemini judgement call.\\n"\n    )\n\n    lines.append("## Runtime Statistics")\n    for key, value in stats.items():\n        lines.append(f"- {key.replace(\'_\', \' \').title()}: {value}")\n\n    return "\\n".join(lines)\n\n\ndef state_to_json(state: SalesState) -> dict:\n    return {\n        "icp": state.icp_analysis or state.icp_raw,\n        "leads": state.qualified_leads,\n        "insufficient_data": state.insufficient_leads,\n        "gaps": state.gaps,\n        "action_log": state.action_log,\n    }\n')
print("✓ sales_agent.py written")


In [ ]:
# Cell 10 — Import the modules we just generated
import importlib
import config, llm, tools, scoring, sales_agent
for m in (config, llm, tools, scoring, sales_agent):
    importlib.reload(m)

from llm import GeminiLLM, GeminiError
from sales_agent import SalesAgent, SalesState, state_to_json
from config import (
    MAX_ITERATIONS, MAX_SEARCH_QUERIES, MAX_SEARCH_RESULTS, MAX_COMPANIES,
    MAX_GEMINI_CALLS_PER_SESSION, GEMINI_MIN_SECONDS_BETWEEN_CALLS,
)

print("✓ Agent modules loaded")
print(f"MAX_ITERATIONS={MAX_ITERATIONS}  MAX_SEARCH_QUERIES={MAX_SEARCH_QUERIES}  "
      f"MAX_SEARCH_RESULTS={MAX_SEARCH_RESULTS}  MAX_COMPANIES={MAX_COMPANIES}")
print(f"Gemini free-tier budget: {MAX_GEMINI_CALLS_PER_SESSION} calls/session, "
      f"paced {GEMINI_MIN_SECONDS_BETWEEN_CALLS}s apart")


In [ ]:
# Cell 11 — Build the Gemini LLM client and the Sales Agent
if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY is not set. Add it under Tools > Secrets and re-run "
        "from Cell 4."
    )

llm_client = GeminiLLM(api_key=GEMINI_API_KEY)
agent = SalesAgent(llm=llm_client, verbose=True)
print(f"✓ Sales Agent ready (model: {llm_client.model})")

In [ ]:
# Cell 12 — Generate data/sample_icps.json
Path("data").mkdir(exist_ok=True)
Path("data/sample_icps.json").write_text('{\n  "saas_india_demo": {\n    "industry": "SaaS",\n    "geography": "India",\n    "company_size": "50-500 employees",\n    "customer_problem": "Operational workflows that could benefit from AI automation",\n    "product": "AI automation platform"\n  },\n  "ecommerce_india_demo": {\n    "industry": "E-commerce",\n    "geography": "India",\n    "company_size": "100-1000 employees",\n    "customer_problem": "Customer support and operational automation",\n    "product": "AI customer-support automation"\n  }\n}\n')
sample_icps = json.loads(Path("data/sample_icps.json").read_text())
print("✓ data/sample_icps.json written")
print(json.dumps(sample_icps, indent=2))

In [ ]:
# Cell 13 — Demo ICP #1: SaaS, India
demo_icp_1 = sample_icps["saas_india_demo"]
demo_icp_1

In [ ]:
# Cell 14 — Run the Sales Agent on Demo ICP #1
Path("outputs").mkdir(exist_ok=True)

state_1 = agent.run(demo_icp_1)

In [ ]:
# Cell 15 — Display ranked leads (Demo 1)
if state_1.qualified_leads:
    for lead in state_1.qualified_leads:
        print(f"{lead['priority']:<20} {lead['score']:>2}/{lead['max_score']}  {lead['company']}  ({lead.get('website','')})")
else:
    print("No leads scored this run — see state_1.gaps for why:", state_1.gaps)

In [ ]:
# Cell 16 — Export outputs/leads.json and outputs/sales_report.md (Demo 1)
Path("outputs/leads.json").write_text(json.dumps(state_to_json(state_1), indent=2))
Path("outputs/sales_report.md").write_text(state_1.final_report)
print("✓ outputs/leads.json written")
print("✓ outputs/sales_report.md written")
print()
print(state_1.final_report[:1500], "...\n[truncated — open outputs/sales_report.md for the full report]")

In [ ]:
# Cell 17 — Demo ICP #2: E-commerce, India — run the agent again
demo_icp_2 = sample_icps["ecommerce_india_demo"]
state_2 = agent.run(demo_icp_2)

for lead in state_2.qualified_leads:
    print(f"{lead['priority']:<20} {lead['score']:>2}/{lead['max_score']}  {lead['company']}  ({lead.get('website','')})")

Path("outputs/leads_demo2.json").write_text(json.dumps(state_to_json(state_2), indent=2))
Path("outputs/sales_report_demo2.md").write_text(state_2.final_report)
print("\n✓ outputs/leads_demo2.json and outputs/sales_report_demo2.md written")

In [ ]:
# Cell 18 — Enter your own custom ICP
custom_icp = {
    "industry": input("Target industry: ") or "SaaS",
    "geography": input("Target geography: ") or "India",
    "company_size": input("Company size (e.g. 50-500 employees): ") or "50-500 employees",
    "customer_problem": input("Ideal customer's problem: ") or "Manual operational workflows",
    "product": input("Your product/service: ") or "AI automation platform",
}
custom_icp

In [ ]:
# Cell 19 — Run the Sales Agent on your custom ICP
state_custom = agent.run(custom_icp)

In [ ]:
# Cell 20 — Display ranked leads and export outputs (custom run)
for lead in state_custom.qualified_leads:
    print(f"{lead['priority']:<20} {lead['score']:>2}/{lead['max_score']}  {lead['company']}  ({lead.get('website','')})")

Path("outputs/leads_custom.json").write_text(json.dumps(state_to_json(state_custom), indent=2))
Path("outputs/sales_report_custom.md").write_text(state_custom.final_report)
print("\n✓ outputs/leads_custom.json and outputs/sales_report_custom.md written")

In [ ]:
# Cell 21 — Runtime statistics (free-tier usage check)
stats = llm_client.stats()
stats["search_calls"] = agent.search_call_count
stats["companies_researched_total"] = agent.companies_researched_count

print("📊 Cumulative runtime statistics across all runs this session:")
for k, v in stats.items():
    print(f"  {k}: {v}")

if stats["gemini_calls_remaining"] == 0:
    print("\n⚠ Session Gemini call budget used up — any further runs will rely on")
    print("  the deterministic keyword fallback instead of fresh Gemini scoring.")
else:
    print(f"\n✓ {stats['gemini_calls_remaining']} Gemini calls left in this session's budget.")


In [ ]:
# Cell 22 — Generate tests/test_scoring.py
# These tests do NOT require Gemini or network access.
Path("tests").mkdir(exist_ok=True)
Path("tests/__init__.py").write_text("")
Path("tests/test_scoring.py").write_text('"""\ntests/test_scoring.py — Lightweight tests that do NOT require a Gemini\nAPI key or network access. Run with: python -m pytest tests/ -v\n(or: python -m unittest discover tests)\n"""\n\nimport os\nimport sys\nimport unittest\n\nsys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\n\nfrom config import MAX_LEAD_SCORE\nfrom scoring import InvalidScoreError, calculate_lead_score, classify_priority, score_lead\nfrom tools import dedupe_companies, normalize_company_name, normalize_domain\nfrom sales_agent import SalesState\n\n\ndef make_criteria(scores):\n    """scores: list of 6 ints or \'unknown\', in SCORING_CRITERIA order."""\n    from config import SCORING_CRITERIA\n\n    return {\n        name: {"score": s, "evidence": "test evidence"}\n        for name, s in zip(SCORING_CRITERIA, scores)\n    }\n\n\nclass TestLeadScoreCalculation(unittest.TestCase):\n    def test_high_fit_lead(self):\n        criteria = make_criteria([3, 3, 3, 3, 3, 3])\n        self.assertEqual(calculate_lead_score(criteria), 18)\n        self.assertEqual(calculate_lead_score(criteria), MAX_LEAD_SCORE)\n\n    def test_medium_fit_lead(self):\n        criteria = make_criteria([2, 2, 2, 2, 2, 2])\n        self.assertEqual(calculate_lead_score(criteria), 12)\n\n    def test_low_fit_lead(self):\n        criteria = make_criteria([1, 1, 1, 1, 1, 2])\n        self.assertEqual(calculate_lead_score(criteria), 7)\n\n    def test_missing_evidence_treated_as_zero(self):\n        criteria = make_criteria(["unknown", "unknown", 3, 3, 2, 1])\n        self.assertEqual(calculate_lead_score(criteria), 9)\n\n    def test_missing_criterion_key_treated_as_zero(self):\n        criteria = make_criteria([3, 3, 3, 3, 3, 3])\n        del criteria["evidence_quality"]\n        self.assertEqual(calculate_lead_score(criteria), 15)\n\n    def test_invalid_score_raises(self):\n        criteria = make_criteria([5, 3, 3, 3, 3, 3])\n        with self.assertRaises(InvalidScoreError):\n            calculate_lead_score(criteria)\n\n    def test_negative_score_raises(self):\n        criteria = make_criteria([-1, 3, 3, 3, 3, 3])\n        with self.assertRaises(InvalidScoreError):\n            calculate_lead_score(criteria)\n\n    def test_non_int_score_raises(self):\n        criteria = make_criteria(["high", 3, 3, 3, 3, 3])\n        with self.assertRaises(InvalidScoreError):\n            calculate_lead_score(criteria)\n\n\nclass TestPriorityClassification(unittest.TestCase):\n    def test_high_priority(self):\n        self.assertEqual(classify_priority(18), "HIGH PRIORITY")\n        self.assertEqual(classify_priority(15), "HIGH PRIORITY")\n\n    def test_medium_priority(self):\n        self.assertEqual(classify_priority(14), "MEDIUM PRIORITY")\n        self.assertEqual(classify_priority(11), "MEDIUM PRIORITY")\n\n    def test_low_priority(self):\n        self.assertEqual(classify_priority(10), "LOW PRIORITY")\n        self.assertEqual(classify_priority(7), "LOW PRIORITY")\n\n    def test_insufficient_data(self):\n        self.assertEqual(classify_priority(6), "INSUFFICIENT DATA")\n        self.assertEqual(classify_priority(0), "INSUFFICIENT DATA")\n\n    def test_out_of_range_raises(self):\n        with self.assertRaises(InvalidScoreError):\n            classify_priority(19)\n        with self.assertRaises(InvalidScoreError):\n            classify_priority(-1)\n\n\nclass TestScoreLead(unittest.TestCase):\n    def test_full_lead_record(self):\n        criteria = make_criteria([3, 3, 2, 3, 2, 3])\n        lead = score_lead("Acme Corp", criteria, sources=["https://acme.com/about"])\n        self.assertEqual(lead["company"], "Acme Corp")\n        self.assertEqual(lead["score"], 16)\n        self.assertEqual(lead["priority"], "HIGH PRIORITY")\n        self.assertEqual(lead["sources"], ["https://acme.com/about"])\n\n\nclass TestDuplicateCompanyHandling(unittest.TestCase):\n    def test_dedupe_by_name(self):\n        companies = [\n            {"company": "Acme Inc.", "website": "https://acme.com", "source_urls": ["https://a.com"]},\n            {"company": "Acme", "website": "https://acme.com", "source_urls": ["https://b.com"]},\n        ]\n        deduped = dedupe_companies(companies)\n        self.assertEqual(len(deduped), 1)\n        self.assertIn("https://b.com", deduped[0]["source_urls"])\n\n    def test_dedupe_by_domain(self):\n        companies = [\n            {"company": "Acme Corp", "website": "https://www.acme.com", "source_urls": []},\n            {"company": "Acme Corporation Pvt Ltd", "website": "https://acme.com/home", "source_urls": []},\n        ]\n        deduped = dedupe_companies(companies)\n        self.assertEqual(len(deduped), 1)\n\n    def test_no_false_positive_dedupe(self):\n        companies = [\n            {"company": "Acme Corp", "website": "https://acme.com", "source_urls": []},\n            {"company": "Zenith Corp", "website": "https://zenith.com", "source_urls": []},\n        ]\n        deduped = dedupe_companies(companies)\n        self.assertEqual(len(deduped), 2)\n\n\nclass TestNormalization(unittest.TestCase):\n    def test_normalize_domain_strips_www(self):\n        self.assertEqual(normalize_domain("https://www.acme.com/about"), "acme.com")\n\n    def test_normalize_domain_no_scheme(self):\n        self.assertEqual(normalize_domain("acme.com"), "acme.com")\n\n    def test_normalize_company_name_strips_suffix(self):\n        self.assertEqual(normalize_company_name("Acme Inc."), "acme")\n        self.assertEqual(normalize_company_name("Acme Private Limited"), "acme")\n\n\nclass TestAgentStateInitialization(unittest.TestCase):\n    def test_default_state(self):\n        state = SalesState(icp_raw={"industry": "SaaS"})\n        self.assertEqual(state.icp_raw["industry"], "SaaS")\n        self.assertEqual(state.plan, [])\n        self.assertEqual(state.candidate_companies, [])\n        self.assertEqual(state.qualified_leads, [])\n        self.assertEqual(state.iterations, 0)\n        self.assertEqual(state.final_report, "")\n\n    def test_state_log(self):\n        state = SalesState(icp_raw={})\n        state.log("ANALYZE_ICP", "test detail")\n        self.assertEqual(len(state.action_log), 1)\n        self.assertEqual(state.action_log[0]["action"], "ANALYZE_ICP")\n\n\nclass TestSearchResultParsing(unittest.TestCase):\n    def test_search_result_shape(self):\n        # Simulated shape returned by tools.web_search — validated here\n        # without hitting the network.\n        fake_result = {"title": "Acme", "url": "https://acme.com", "snippet": "Acme is a SaaS company."}\n        self.assertIn("title", fake_result)\n        self.assertIn("url", fake_result)\n        self.assertIn("snippet", fake_result)\n\n\nif __name__ == "__main__":\n    unittest.main()\n')
print("✓ tests/test_scoring.py written")

In [ ]:
# Cell 23 — Run the unit tests
!python -m unittest discover tests -v

In [ ]:
# Cell 24 — Generate the remaining GitHub-ready project files
Path("requirements.txt").write_text('google-genai>=0.3.0\nddgs>=6.0.0\nrequests>=2.31.0\nbeautifulsoup4>=4.12.0\n')
Path(".env.example").write_text('# Copy this file to .env for local (non-Colab) use, or add these as\n# Google Colab Secrets (Tools > Secrets) when running the notebook.\n\n# Required: your Gemini free-tier API key.\nGEMINI_API_KEY=your-gemini-api-key-here\n\n# Optional: override the default Gemini model.\nGEMINI_MODEL=gemini-2.0-flash\n')
Path("outputs").mkdir(exist_ok=True)
Path("outputs/.gitkeep").write_text("")

print("✓ requirements.txt written")
print("✓ .env.example written")
print("✓ outputs/.gitkeep written")

In [ ]:
# Cell 25 — Generate README.md
Path("README.md").write_text('# 📈 Agentic Sales Agent\n\nProject 3 of a four-project Agentic AI portfolio (Research Agent → Coding Agent → **Sales Agent** → Recruiting Agent). Runs entirely in Google Colab on the Gemini free tier.\n\n## Overview\n\nAn autonomous **B2B sales research and lead-qualification assistant**. Give it an Ideal Customer Profile (ICP) and it plans a research strategy, searches the public web, extracts candidate companies, fetches their public pages, scores them against your ICP with transparent, evidence-backed criteria, and produces a ranked research report.\n\n## Problem\n\nManually researching whether a company fits an ICP means opening a dozen tabs, reading marketing copy, and guessing at fit. It\'s slow and inconsistent, and it\'s easy to lose track of *why* a company seemed like a good fit.\n\n## Solution\n\nAn agent that treats lead research as a multi-step workflow — not a single prompt. It plans what to search for, discovers candidates, gathers public evidence, scores each company against explicit criteria, and cites its sources for every score.\n\n## Why This Is Agentic\n\nThis is **not** `User → Gemini → List of companies`. It is:\n\n```\nICP → ICP Analyzer → Planner → Market Search → Company Discovery\n    → Company Research → Qualification → Lead Scoring\n    → Prioritization → Final Report\n```\n\nThe agent maintains explicit state (`SalesState`), chooses actions from a fixed action set, runs local (free) tools between LLM calls, and loops — checking after each round whether it has enough evidence or needs to research further — inside a hard iteration cap.\n\n## Architecture\n\n```\n                    USER ICP\n                       │\n                       ▼\n              ┌─────────────────┐\n              │ ICP ANALYZER    │  (Gemini call 1)\n              └────────┬────────┘\n                       ▼\n              ┌─────────────────┐\n              │    PLANNER      │  (deterministic)\n              └────────┬────────┘\n                       ▼\n             ┌──────────────────┐\n             │ MARKET SEARCH    │  (ddgs, free, local)\n             └────────┬─────────┘\n                      ▼\n             ┌──────────────────┐\n             │ COMPANY DISCOVERY│  (Gemini call 2)\n             └────────┬─────────┘\n                      ▼\n             ┌──────────────────┐\n             │ COMPANY RESEARCH │  (requests + bs4, local)\n             └────────┬─────────┘\n                      ▼\n             ┌──────────────────┐\n             │ QUALIFICATION    │  (Gemini call 3)\n             └────────┬─────────┘\n                      ▼\n             ┌──────────────────┐\n             │ LEAD SCORING     │  (deterministic, scoring.py)\n             └────────┬─────────┘\n                      ▼\n             ┌──────────────────┐\n             │ PRIORITIZATION   │  (deterministic)\n             └────────┬─────────┘\n                      ▼\n                FINAL REPORT       (Gemini call 4, synthesis + Markdown)\n```\n\n## Agent Loop\n\n```\nwhile iterations < MAX_ITERATIONS:\n    inspect state\n    decide next action\n    execute search / research\n    collect observations\n    qualify companies\n    check whether enough evidence exists\n    if enough evidence: finish\n    else: research more (bounded — never infinite)\n```\n\n`MAX_ITERATIONS = 4`. The loop always terminates, either because enough HIGH/MEDIUM priority leads were found, or because the iteration cap was hit — in which case the report is still generated with whatever evidence exists, and gaps are disclosed.\n\n### Agent actions\n\n`ANALYZE_ICP`, `GENERATE_SEARCH_QUERIES`, `SEARCH_MARKET`, `RESEARCH_COMPANY`, `QUALIFY_LEADS`, `IDENTIFY_GAPS`, `FINISH`\n\n## ICP Analysis\n\nGemini turns a raw ICP (industry, geography, size, problem, product) into a structured strategy: target criteria, relevant public signals to look for, generated search terms, and explicit qualification/disqualification criteria — before any search happens.\n\n## Lead Discovery\n\n`ddgs` (free DuckDuckGo search) runs up to `MAX_SEARCH_QUERIES` targeted queries generated from the ICP. Results are batched into a single Gemini call that extracts distinct candidate companies (not directories or "top 10" articles), deduplicated by normalized name and domain.\n\n## Company Research\n\nFor each candidate, the agent fetches a small set of public pages (`homepage`, `/about`, `/product`, etc., capped at `MAX_COMPANY_PAGES`) using `requests` + `beautifulsoup4`. No login-gated content, no aggressive crawling, no personal data.\n\n## Lead Qualification\n\nEvery company is scored 0–3 on six criteria — industry fit, geography fit, company size fit, problem fit, technology fit, evidence quality — **only from the public text actually retrieved**. If the text doesn\'t support a criterion, it\'s marked `unknown`, never guessed.\n\n## Lead Scoring\n\nScoring is **deterministic Python** (`scoring.py`), not left to the LLM to total up:\n\n```python\ndef calculate_lead_score(criteria):\n    ...  # sums 6 criteria, 0-3 each, "unknown" counts as 0\n```\n\n| Score | Priority |\n|---|---|\n| 15–18 | HIGH PRIORITY |\n| 11–14 | MEDIUM PRIORITY |\n| 7–10 | LOW PRIORITY |\n| < 7 | INSUFFICIENT DATA |\n\n## Evidence Tracking\n\nEvery criterion score carries a one-line evidence string and every lead carries its source URLs. Sales angles are only generated when there is supporting public evidence, and are phrased cautiously ("explore whether...") — never as certainties.\n\n## Project Structure\n\n```\n03-sales-agent/\n│\n├── README.md\n├── sales_agent.py        # state, agent loop, ICP/discovery/research/qualification, report builder\n├── tools.py               # free web search, page fetching, dedupe/normalize helpers\n├── llm.py                 # Gemini wrapper (google-genai), JSON-safe parsing, call tracking\n├── config.py               # all tunable limits and model name\n├── scoring.py              # deterministic scoring + priority classification\n├── requirements.txt\n├── .env.example\n│\n├── data/\n│   └── sample_icps.json\n│\n├── outputs/\n│   └── .gitkeep\n│\n└── tests/\n    └── test_scoring.py\n```\n\n```\nnotebooks/\n└── Agentic_Sales_Agent_Colab.ipynb\n```\n\n## Tech Stack\n\n- Python 3\n- `google-genai` (Gemini free tier)\n- `ddgs` (free web search)\n- `requests` + `beautifulsoup4` (public page extraction)\n- No LangChain / LangGraph / CrewAI / AutoGen — the agent loop, state, and tool orchestration are implemented from first principles.\n\n## Google Colab Setup\n\n1. Open the notebook in Google Colab.\n2. Run the dependency install cell.\n3. Add your Gemini key: **Tools → Secrets → add `GEMINI_API_KEY`**, and grant this notebook access.\n4. Run all cells top to bottom.\n\n## Gemini API Setup\n\n1. Get a free-tier key from [Google AI Studio](https://aistudio.google.com/).\n2. Store it as a Colab Secret named `GEMINI_API_KEY` — never hard-code it in a cell.\n3. `GEMINI_MODEL` defaults to `gemini-2.0-flash` and can be overridden (env var or Colab Secret) if a different model is available to your account.\n\n## Running the Demo\n\nThe notebook includes two ready-made ICPs (SaaS/India and E-commerce/India, from `data/sample_icps.json`) and a prompt for a fully custom ICP. Each run prints a live agent trace, then writes:\n\n- `outputs/leads.json` — structured lead data\n- `outputs/sales_report.md` — human-readable report\n\n## Example Output\n\n```\n🤖 AGENTIC SALES AGENT\n🎯 Ideal Customer Profile\nIndustry: SaaS\nGeography: India\n...\n🧠 Analyzing ICP...\n✓ ICP analyzed\n🔎 Market Search 1/4\n✓ 5 results found\n🏢 Candidate companies discovered: 8\n🔍 Researching: Example Company\n✓ Public company information collected\n📊 Qualifying leads...\n✓ Lead score generated\n🔍 Checking information gaps...\n✓ Sufficient evidence collected\n📈 SALES RESEARCH REPORT\n```\n\n## Free-Tier Optimization\n\n- Targets **2–4 Gemini calls per run**: ICP analysis, company discovery (batched), qualification (batched across all companies), final synthesis.\n- All search, page-fetching, and scoring/prioritization logic runs locally with no LLM calls.\n- Hard caps on iterations, queries, results, companies, and pages fetched (`config.py`).\n- Runtime statistics (Gemini calls, search calls, companies discovered/researched, iterations) are printed and included in the report.\n\n## Responsible AI\n\nThis is a **sales research assistant**, not an autonomous salesperson. It does not:\n\n- send outreach (email, LinkedIn, or otherwise)\n- scrape private or login-gated information\n- make decisions about individuals\n- infer sensitive personal characteristics\n- guarantee lead quality\n\nIt only analyzes **publicly available business information** and always requires **human review before any sales outreach**.\n\n## Security and Privacy\n\nThe agent does not collect personal email addresses, private phone numbers, home addresses, passwords, authentication information, or other sensitive personal data. It focuses exclusively on companies and public business information.\n\n## Limitations\n\n- Public web coverage is incomplete — many real companies won\'t surface from a handful of searches.\n- Company size, revenue, and tech stack are often not publicly disclosed and will be marked `unknown`.\n- Website structures vary; some public pages won\'t be found at the guessed paths.\n- Scores reflect apparent fit from public marketing content, not verified qualification.\n- Free-tier rate limits may cause occasional Gemini call failures — the agent degrades gracefully (deterministic fallbacks) rather than crashing.\n\n## Future Improvements\n\n- Pluggable search backends (e.g. a paid API) behind the same `tools.web_search` interface.\n- Caching researched companies across runs to reduce repeat fetches.\n- A lightweight review UI for a human to approve/reject leads before any handoff.\n- Multi-language public page support.\n\n## Author\n\nBuilt as Project 3 of a four-project Agentic AI portfolio, developed and run entirely in Google Colab on the Gemini free tier.\n')
print("✓ README.md written")

In [ ]:
# Cell 26 — Verify the GitHub-ready project structure
expected_files = [
    "README.md", "sales_agent.py", "tools.py", "llm.py", "config.py",
    "scoring.py", "requirements.txt", ".env.example",
    "data/sample_icps.json", "outputs/.gitkeep", "tests/test_scoring.py",
]

print("Project structure check:\n")
all_ok = True
for f in expected_files:
    exists = Path(f).exists()
    all_ok &= exists
    print(f"  {'✓' if exists else '✗ MISSING'}  {f}")

print("\n✅ All expected files present." if all_ok else "\n⚠ Some files are missing — re-run earlier cells.")

In [ ]:
# Cell 27 — Package everything into a GitHub-ready ZIP
zip_root = "03-sales-agent"
files_to_zip = [
    "README.md", "sales_agent.py", "tools.py", "llm.py", "config.py",
    "scoring.py", "requirements.txt", ".env.example",
    "data/sample_icps.json", "outputs/.gitkeep",
    "tests/test_scoring.py", "tests/__init__.py",
]

zip_path = "03-sales-agent.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_zip:
        if Path(f).exists():
            zf.write(f, arcname=f"{zip_root}/{f}")

print(f"✓ {zip_path} created ({Path(zip_path).stat().st_size} bytes)")

In [ ]:
# Cell 28 — Download the ZIP (Colab only)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab — find the file at:", Path(zip_path).resolve())

## GitHub Upload Instructions

1. Unzip `03-sales-agent.zip`.
2. Create a new GitHub repository (e.g. `agentic-ai-portfolio`), or reuse the
   one from Projects 1 and 2.
3. Place the unzipped `03-sales-agent/` folder alongside `01-research-agent/`
   and `02-coding-agent/` in the repo root, and add this notebook under
   `notebooks/Agentic_Sales_Agent_Colab.ipynb`.
4. `git add . && git commit -m "Add Project 3: Agentic Sales Agent" && git push`

**Reminder:** this agent produces research candidates, not verified leads.
Always have a human review `outputs/sales_report.md` before any outreach.